# Transfer Learning pour la Classification d'Images

Ce notebook entraîne un modèle de classification d'images en utilisant le transfer learning avec ResNet50 ou EfficientNet.


In [ ]:
import sys
import os
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src.dataset import load_cifar10_dataset, load_imagefolder_dataset
from src.utils import calculate_accuracy, save_checkpoint, plot_training_history, plot_confusion_matrix, print_classification_report, get_model_summary

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. Chargement des Données


In [ ]:
# Configuration
USE_CIFAR10 = True  # Mettre à False pour utiliser un dataset local
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2

if USE_CIFAR10:
    train_loader, val_loader, class_names = load_cifar10_dataset(
        data_dir='../data',
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS
    )
else:
    train_loader, val_loader, class_names = load_imagefolder_dataset(
        data_dir='../data',
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS
    )

print(f"Nombre de classes: {len(class_names)}")
print(f"Classes: {class_names}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")


## 2. Création du Modèle avec Transfer Learning


In [ ]:
# Configuration du modèle
MODEL_NAME = 'resnet50'  # 'resnet50' ou 'efficientnet'
NUM_CLASSES = len(class_names)
PRETRAINED = True
FREEZE_BACKBONE = True  # Geler le backbone initialement
UNFREEZE_LAST_N = 0  # Débloquer les N dernières couches (0 = tout gelé si FREEZE_BACKBONE=True)

def create_model(model_name, num_classes, pretrained, freeze_backbone, unfreeze_last_n):
    """Créer le modèle avec transfer learning"""
    if model_name.lower() == 'resnet50':
        from torchvision.models import resnet50, ResNet50_Weights
        
        if pretrained:
            model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        else:
            model = resnet50(weights=None)
        
        # Remplacer le classifieur
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
        
        # Geler le backbone si demandé
        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False
            
            # Débloquer les N dernières couches
            if unfreeze_last_n > 0:
                layers = list(model.children())
                for layer in layers[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True
        
        # Toujours débloquer le classifieur
        for param in model.fc.parameters():
            param.requires_grad = True
    
    elif model_name.lower() == 'efficientnet':
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        
        if pretrained:
            model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        else:
            model = efficientnet_b0(weights=None)
        
        # Remplacer le classifieur
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
        
        # Geler le backbone si demandé
        if freeze_backbone:
            for param in model.features.parameters():
                param.requires_grad = False
            
            # Débloquer les N dernières couches
            if unfreeze_last_n > 0:
                layers = list(model.features.children())
                for layer in layers[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True
        
        # Toujours débloquer le classifieur
        for param in model.classifier.parameters():
            param.requires_grad = True
    
    return model.to(device)

# Créer le modèle
model = create_model(MODEL_NAME, NUM_CLASSES, PRETRAINED, FREEZE_BACKBONE, UNFREEZE_LAST_N)
get_model_summary(model, (3, IMG_SIZE, IMG_SIZE))


## 3. Configuration de l'Entraînement


In [ ]:
# Hyperparamètres
EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
USE_AMP = False  # Mixed Precision Training (accélère l'entraînement)

# Loss et Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

# Mixed Precision
scaler = GradScaler() if USE_AMP else None

print(f"Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Weight Decay: {WEIGHT_DECAY}")
print(f"  Mixed Precision: {USE_AMP}")


## 4. Boucle d'Entraînement


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, use_amp=False, scaler=None):
    """Entraîner pour une époque"""
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    
    pbar = tqdm(train_loader, desc='Training')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item()
        acc = calculate_accuracy(outputs, labels)
        running_acc += acc
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc:.2f}%'})
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = running_acc / len(train_loader)
    
    return epoch_loss, epoch_acc

def validate(model, val_loader, criterion, device, use_amp=False):
    """Valider le modèle"""
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            
            if use_amp:
                with autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            acc = calculate_accuracy(outputs, labels)
            running_acc += acc
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{acc:.2f}%'})
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = running_acc / len(val_loader)
    
    return epoch_loss, epoch_acc, all_preds, all_labels


In [ ]:
# Historique d'entraînement
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_acc = 0.0
os.makedirs('../models', exist_ok=True)

# Boucle d'entraînement
print(f"\n{'='*60}")
print("Démarrage de l'entraînement...")
print(f"{'='*60}\n")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print("-" * 60)
    
    # Entraîner
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, USE_AMP, scaler)
    
    # Valider
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device, USE_AMP)
    
    # Mettre à jour le learning rate
    scheduler.step(val_loss)
    
    # Mettre à jour l'historique
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Sauvegarder le meilleur modèle
    is_best = val_acc > best_acc
    if is_best:
        best_acc = val_acc
    
    checkpoint_path = f'../models/checkpoint_epoch_{epoch+1}.pth'
    save_checkpoint({
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'best_acc': best_acc,
        'val_acc': val_acc,
        'class_names': class_names,
        'model_name': MODEL_NAME,
        'num_classes': NUM_CLASSES
    }, checkpoint_path, is_best)
    
    print(f"\nEpoch {epoch+1} Résumé:")
    print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"  Meilleure Val Acc: {best_acc:.2f}%")

print(f"\n{'='*60}")
print("Entraînement terminé!")
print(f"Meilleure précision de validation: {best_acc:.2f}%")
print(f"Modèle sauvegardé dans: ../models/best_model.pth")


## 5. Visualisation des Résultats d'Entraînement


In [ ]:
# Afficher les courbes de loss et accuracy
plot_training_history(history, save_path='../models/training_history.png')


## 6. Matrice de Confusion et Rapport de Classification


In [ ]:
# Charger le meilleur modèle pour l'évaluation finale
checkpoint = torch.load('../models/best_model.pth', map_location=device)
model.load_state_dict(checkpoint['state_dict'])

# Évaluer sur le validation set
val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device, USE_AMP)

# Matrice de confusion
plot_confusion_matrix(np.array(val_labels), np.array(val_preds), class_names,
                     save_path='../models/confusion_matrix.png', normalize=True)

# Rapport de classification
print_classification_report(np.array(val_labels), np.array(val_preds), class_names)


## 7. Test d'Inférence sur des Images

Testons le modèle sur quelques images du validation set.


In [ ]:
# Fonction pour dénormaliser
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor

# Tester sur quelques images
model.eval()
data_iter = iter(val_loader)
images, labels = next(data_iter)
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
    outputs = model(images)
    probabilities = torch.nn.functional.softmax(outputs, dim=1)
    _, preds = torch.max(outputs, 1)

# Visualiser les prédictions
num_images = 12
fig, axes = plt.subplots(3, 4, figsize=(15, 12))
axes = axes.ravel()

for idx in range(num_images):
    img = images[idx]
    true_label = labels[idx].item()
    pred_label = preds[idx].item()
    prob = probabilities[idx][pred_label].item()
    
    # Dénormaliser
    img_denorm = denormalize(img.clone().cpu())
    img_denorm = torch.clamp(img_denorm, 0, 1)
    img_np = img_denorm.permute(1, 2, 0).cpu().numpy()
    
    axes[idx].imshow(img_np)
    color = 'green' if true_label == pred_label else 'red'
    axes[idx].set_title(f'True: {class_names[true_label]}\nPred: {class_names[pred_label]} ({prob:.2f})',
                       color=color, fontsize=10, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Prédictions sur le Validation Set', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Résumé

L'entraînement est terminé! Le modèle a été sauvegardé dans `../models/best_model.pth`.

**Prochaines étapes:**
- Utiliser `src/infer.py` pour faire des prédictions sur de nouvelles images
- Lancer l'application Streamlit avec `streamlit run app/app.py`
- Convertir le modèle en ONNX avec `scripts/convert_to_onnx.py` pour un déploiement léger
